In [1]:
!pip install biopython
!pip install numpy
!pip install pandas
!pip install scipy
!pip install openpyxl
import numpy as np
from Bio.Seq import Seq
import pandas as pd
import itertools
import re
from scipy.spatial import distance
import json
from Bio.SeqUtils import MeltingTemp as mt
import random
import pprint
import string

In [2]:
#Paste gene
gene = 'ATGgcggttctctggaggctgagtgccgtttgcggtgccctaggaggccgagctctgttgcttcgaactccagtggtcagacctgctcatatctcagcatttcttcaggaccgacctatcccagaatggtgtggagtgcagcacatacacttgtcaccgagccaccattctggctccaaggctgcatctctccactggactagcgagagggttgtcagtgttttgctcctgggtctgcttccggctgcttatttgaatccttgctctgcgatggactattccctggctgcagccctcactcttcatggtcactggggccttggacaagttgttactgactatgttcatggggatgccttgcagaaagctgccaaggcagggcttttggcactttcagctttaacctttgctgggctttgctatttcaactatcacgatgtgggcatctgcaaagctgttgccatgctgtggaagctcacg'
gene = gene.upper()
gene = Seq(gene)


In [3]:
#Get block size and initial breakpoint size
gene_size = len(gene)
block_size_range = [165, 175]
max_oligo_size=250
slack = 5
block_size = block_size_range[0] + np.argmin(
    [abs(gene_size/(i+block_size_range[0])-round(gene_size/(i+block_size_range[0]))) \
         for i in range(0, block_size_range[1]-block_size_range[0])])
fragment_number = int(np.ceil(gene_size/block_size))
first_overlap = gene[0:3]
last_overlap = Seq('ACG')
first_breakpoint = 0
last_breakpoint = gene_size-3

initial_breakpoints = [(first_breakpoint,block_size+slack+2,last_breakpoint)] + \
                        [(0,i-2-slack,i+block_size+slack+2,last_breakpoint) for i in range(block_size, gene_size-block_size, block_size)] + \
                        [(0,gene_size-2-slack-block_size,last_breakpoint)]


In [4]:
#Primer set from Rama Ranganathan paper
orthogonal_primers = pd.read_excel('./ysx008_supp_st_1.xlsx')
orthogonal_primers = orthogonal_primers[orthogonal_primers['Keep primer in orthogonal set?'] == 'Yes']['Sequence'].values
random.seed(42)
random.shuffle(orthogonal_primers)
orthogonal_F = [Seq(s) for s in orthogonal_primers[:len(orthogonal_primers)//2]]
orthogonal_R = [Seq(s) for s in orthogonal_primers[len(orthogonal_primers)//2:2*(len(orthogonal_primers)//2)]]


In [5]:
#Print primers in plate format to order via IDT
well_plate = []
for number in list(range(1,1+12)):
    well_plate.extend([letter+str(number) for letter in list(string.ascii_uppercase)[0:8]])
nextera_R1 = Seq('TCGTCGGCAGCGTCAGATGTGTATAAGAGACAG')
nextera_R2 = Seq('GTCTCGTGGGCTCGGAGATGTGTATAAGAGACAG')
for i,primer in enumerate(orthogonal_F):
    print(well_plate[i] + '\t' + 'RR_OrthogonalFv1_' + str(i+1) + '\t' + str(nextera_R1+primer))
    
for i,primer in enumerate(orthogonal_R):
    print(well_plate[i] + '\t' + 'RR_OrthogonalRv1_' + str(i+1) + '\t' + str(nextera_R2+primer))

A1	RR_OrthogonalFv1_1	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGAGTATCTCAGCAAGGGCAAC
B1	RR_OrthogonalFv1_2	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGCCAGAGCTTAGGGGACATAC
C1	RR_OrthogonalFv1_3	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGGCACGCAAAAGGACATAACC
D1	RR_OrthogonalFv1_4	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGAGACACAAGGCTGATTCCAG
E1	RR_OrthogonalFv1_5	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGTCCAATTATACGGAGCAGGC
F1	RR_OrthogonalFv1_6	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGTAGTTGAGAACACGAACCCG
G1	RR_OrthogonalFv1_7	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGCAGACCTACGGATCTTAGCG
H1	RR_OrthogonalFv1_8	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGAAGGCCCAGAAGGATACAAC
A2	RR_OrthogonalFv1_9	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGTATCAATCCGGAACCAGTGC
B2	RR_OrthogonalFv1_10	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGAGTCCGACACAATGTGACAC
C2	RR_OrthogonalFv1_11	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGACGAGATGATGCACCGATAG
D2	RR_OrthogonalFv1_12	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGGACCATGCAAGGAGAGGTAC
E2	RR_OrthogonalFv1_13	TCGTCGGCAGCGTCAGATGTGTATAAGAGACAGTGCATAGTATCCCAACAGGG
F2	RR_Or

In [6]:
#Import SapI data
sapI_overlap_empirical = pd.read_csv('./sapI_empirical.csv')
sapI_overlap_empirical.index = sapI_overlap_empirical['Overhang']
sapI_overlap_empirical = sapI_overlap_empirical.drop(columns=['Overhang'])
sapI_overlap_empirical = sapI_overlap_empirical + 1


In [7]:
#Import information about codon usage for mutagenesis
codons_ranked_by_usage = {
    "A": ["GCC", "GCT", "GCA", "GCG"],
    "C": ["TGC", "TGT"],
    "D": ["GAC", "GAT"],
    "E": ["GAG", "GAA"],
    "F": ["TTC", "TTT"],
    "G": ["GGC", "GGA", "GGG", "GGT"],
    "H": ["CAC", "CAT"],
    "I": ["ATC", "ATT", "ATA"],
    "K": ["AAG", "AAA"],
    "L": ["CTG", "CTC", "CTT", "TTG", "TTA", "CTA"],
    "M": ["ATG"],
    "N": ["AAC", "AAT"],
    "P": ["CCC", "CCT", "CCA", "CCG"],
    "Q": ["CAG", "CAA"],
    "R": ["CGG", "AGA", "AGG", "CGC", "CGA", "CGT"],
    "S": ["AGC", "TCC", "TCT", "AGT", "TCA", "TCG"],
    "T": ["ACC", "ACA", "ACT", "ACG"],
    "V": ["GTG", "GTC", "GTT", "GTA"],
    "W": ["TGG"],
    "Y": ["TAC", "TAT"],
}


In [8]:
#Set blacklist of inefficient codons
codon_blacklist = []
for codon in sapI_overlap_empirical.index.values:
    if sapI_overlap_empirical.loc[Seq(codon),Seq(codon).reverse_complement()] < 2000:
        codon_blacklist.append(codon)
codon_blacklist


['TTT',
 'TTG',
 'TGG',
 'CGG',
 'TCG',
 'GCG',
 'CCG',
 'CGC',
 'TTA',
 'TGA',
 'CGA',
 'TCA',
 'CCA',
 'TAA',
 'CAA',
 'AAA']

In [9]:
def build_kmers(sequence, 
                ksize):
    kmers = []
    n_kmers = len(sequence) - ksize + 1

    for i in range(n_kmers):
        kmer = sequence[i:i + ksize]
        kmers.append(kmer)

    return kmers

def compute_overlaps(breakpoints, 
                     inclusion_array, 
                     gene=gene):
    
    overlaps = [[gene[val:val+3].reverse_complement(), gene[val:val+3]] for val in breakpoints]
    counter = 0
    for val in inclusion_array:
        if val == -1:
            (overlaps[counter][1],overlaps[counter+1][0]) = (overlaps[counter+1][0],overlaps[counter][1])
            counter += 1
        elif val == 0:
            overlaps[counter][1] = overlaps[counter+1][1]
            del overlaps[counter+1]
        
    return overlaps

def score_breakpoints(gene, 
                      breakpoint_pair, 
                      empirical, 
                      codon_blacklist=codon_blacklist):
    
    #subset empirical matrix by the set of all overlaps
    all_overlaps = []
    for breakpoint in breakpoint_pair:
        all_overlaps.append(gene[breakpoint:(breakpoint+3)])
        all_overlaps.append(gene[breakpoint:(breakpoint+3)].reverse_complement())
    all_overlaps = [str(o) for o in all_overlaps]
    if (len(np.unique(all_overlaps)) == len(all_overlaps)) & (len(set(all_overlaps).intersection(set(codon_blacklist))) == 0):
        empirical_subset = empirical.loc[all_overlaps,all_overlaps]

        #compute fidelity score
        empirical_subset = empirical_subset/empirical_subset.sum(axis=1)
        fidelity_score = 1
        for breakpoint in breakpoint_pair:
            fidelity_score = fidelity_score * empirical_subset.loc[gene[breakpoint:(breakpoint+3)],gene[breakpoint:(breakpoint+3)].reverse_complement()]
    
    else:
        fidelity_score = 0
    
    return fidelity_score
    
def optimize_breakpoints(gene, 
                         breakpoint_pair, 
                         indices_to_shift, 
                         indices_of_array,
                         slack, 
                         empirical=sapI_overlap_empirical, 
                         codon_blacklist=codon_blacklist):
    
    #compute all enrichments
    shifts = list(range(-slack,slack+1))
    if (len(indices_to_shift) > 2) | (len(indices_to_shift) < 1):
        print('Error -- too many or too few breakpoints!')
        optimum_breakpoint = breakpoint_pair
        optimum_score = 0
    elif (len(indices_to_shift) == 1): #external pair
        scores = [0]*len(shifts)
        for i,shift in enumerate(shifts):
            scores[i] = score_breakpoints(gene, breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shift] + breakpoint_pair[(indices_to_shift[0]+1):], 
                                          empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
        
        optimum_shift = np.argmax(scores)
        optimum_breakpoint = breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shifts[optimum_shift]] + breakpoint_pair[(indices_to_shift[0]+1):]
        optimum_score = scores[optimum_shift]
        optimum_length = optimum_breakpoint[indices_of_array[1]] - optimum_breakpoint[indices_of_array[0]]
            
            
    else: #internal pair
        indices_to_shift = sorted(indices_to_shift)
        scores = np.zeros((len(shifts),len(shifts)))
        for i,shift1 in enumerate(shifts):
            for j,shift2 in enumerate(shifts):
                scores[i,j] = score_breakpoints(gene, breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shift1] + \
                                                            breakpoint_pair[(indices_to_shift[0]+1):indices_to_shift[1]] + \
                                                            [breakpoint_pair[indices_to_shift[1]]+shift2] + breakpoint_pair[(indices_to_shift[1]+1):], 
                                              empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
                
        optimum_shift = np.unravel_index(np.argmax(scores,axis=None), scores.shape)
        optimum_breakpoint = breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shifts[optimum_shift[0]]] + \
                                            breakpoint_pair[(indices_to_shift[0]+1):indices_to_shift[1]] + \
                                            [breakpoint_pair[indices_to_shift[1]]+shifts[optimum_shift[1]]] + breakpoint_pair[(indices_to_shift[1]+1):]
        optimum_score = scores[optimum_shift]
        optimum_length = optimum_breakpoint[indices_of_array[1]] - optimum_breakpoint[indices_of_array[0]] + 3
    
    return optimum_breakpoint, optimum_score, optimum_length

def optimize_gene(gene, 
                  block_size_range=block_size_range, 
                  slack=slack, 
                  empirical=sapI_overlap_empirical, 
                  codon_blacklist=codon_blacklist): 
    
    #setup initial inputs to optimization
    gene_size = len(gene)
    protein_size = len(gene.translate())
        
    #exclude gene if it is too big
    if protein_size > 1000:
        print('Protein size too big!')
        
    #divide genes between 500 and 1000aa into two blocks
    elif protein_size > 550:
        print('Protein size too big! Will add two superblock (551aa+ proteins) soon.')
        
        
#         print('Protein is two superblocks.')
#         #set up superblocks, now 5-part SapI 
#         #find intermediate codon which can be thrown out of library and be used as golden-gate overlap
#         first_breakpoint = 0
#         last_breakpoint = gene_size-3
#         shifts = list(range(-slack,slack+1))
#         score_breaks = [0]*len(shifts)
#         for i,shift in enumerate(shifts):
#             codon_loc = 3*(protein_size//2+shift)
#             break_codon = gene[codon_loc:(codon_loc+3)]
#             score_breaks[i] = score_breakpoints(gene, [first_breakpoint, codon_loc, last_breakpoint], 
#                                           empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
        
#         optimum_shift = np.argmax(score_breaks)
#         optimum_codontobreak = 3*(protein_size//2+shifts[optimum_shift])
#         gene_first_half = gene[0:optimum_codontobreak+3]
#         gene_second_half = gene[optimum_codontobreak:]
        
#         optimum_breakpoints = []
#         optimum_scores = []
#         optimum_lengths = []
#         oligo_array_indices = []
        
#         for i, gene_block in enumerate([gene_first_half,gene_second_half]):
#             #now, set up blocks for each half and perform two optimizations
#             gene_block_size = len(gene_block)
#             block_size = block_size_range[0] + np.argmin(
#                 [abs(gene_block_size/(i+block_size_range[0])-round(gene_block_size/(i+block_size_range[0]))) \
#                      for i in range(0, block_size_range[1]-block_size_range[0])])
#             fragment_number = int(np.ceil(gene_block_size/block_size))
#             if i == 0:
#                 pre_first_breakpoint = []
#                 first_breakpoint = [0]
#                 last_breakpoint = [optimum_codontobreak]
#                 post_last_breakpoint = [gene_size-3]
#             elif i == 1:
#                 pre_first_breakpoint = [0]
#                 first_breakpoint = [optimum_codontobreak]
#                 last_breakpoint = [gene_size-3]
#                 post_last_breakpoint = []
#             else:
#                 print('Too many superblocks!')
#             initial_breakpoints = [pre_first_breakpoint + first_breakpoint + \
#                                    [first_breakpoint[0]+block_size+slack+2] + last_breakpoint + post_last_breakpoint] + \
#                                 [pre_first_breakpoint + first_breakpoint + \
#                                  [first_breakpoint[0]+j-2-slack, first_breakpoint[0]+j+block_size+slack+2] + last_breakpoint + post_last_breakpoint \
#                                      for j in range(block_size, gene_block_size-block_size-slack-3, block_size)] + \
#                                 [pre_first_breakpoint + first_breakpoint + \
#                                  [first_breakpoint[0]+gene_block_size-2-slack-block_size] + last_breakpoint + post_last_breakpoint]
            
#             #optimize each breakpoint
#             for k,breakpoint in enumerate(initial_breakpoints):
#                 if len(breakpoint) == 4:
#                     indices_of_array = [0+i, 1+i] if k==0 else [1+i, 2+i]
#                     optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1+i], indices_of_array,
#                                                                 slack, empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
#                     optimum_breakpoints.append(optimum_breakpoint)
#                     optimum_scores.append(optimum_score)
#                     optimum_lengths.append(optimum_length)
#                     oligo_array_indices.append(indices_of_array)
                    
#                 else:
#                     indices_of_array = [1+i, 2+i]
#                     optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1+i, 2+i], indices_of_array, 
#                                                                     slack, empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
#                     optimum_breakpoints.append(optimum_breakpoint)
#                     optimum_scores.append(optimum_score)
#                     optimum_lengths.append(optimum_length)
#                     oligo_array_indices.append(indices_of_array)
        
    else:
        #gene is one superblock, 4-part SapI 
        #print('Protein is one superblock.')
        block_size = block_size_range[0] + np.argmin(
            [abs(gene_size/(i+block_size_range[0])-round(gene_size/(i+block_size_range[0]))) \
                 for i in range(0, block_size_range[1]-block_size_range[0])])
        fragment_number = int(np.ceil(gene_size/block_size))
        first_breakpoint = 0
        last_breakpoint = gene_size-3
        initial_breakpoints = [[first_breakpoint,block_size+slack+2,last_breakpoint]] + \
                                [[0,i-2-slack,i+block_size+slack+2,last_breakpoint] for i in range(block_size, gene_size-block_size-slack-3, block_size)] + \
                                [[0,gene_size-2-slack-block_size,last_breakpoint]]
    
        #optimize each breakpoint
        optimum_breakpoints = []
        optimum_scores = []
        optimum_lengths = []
        oligo_array_indices = []
        for k,breakpoint in enumerate(initial_breakpoints):
            if len(breakpoint) == 3:
                indices_of_array = [0, 1] if k==0 else [1, 2]
                optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1], indices_of_array,
                                                            slack, empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
                optimum_breakpoints.append(optimum_breakpoint)
                optimum_scores.append(optimum_score)
                optimum_lengths.append(optimum_length)
                oligo_array_indices.append(indices_of_array)
            else:
                indices_of_array = [1, 2]
                optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1, 2], indices_of_array,
                                                            slack, empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
                optimum_breakpoints.append(optimum_breakpoint)
                optimum_scores.append(optimum_score)
                optimum_lengths.append(optimum_length)
                oligo_array_indices.append(indices_of_array)
    
    optimum_overlaps = [[str(gene[t:(t+3)]) for t in s] for s in optimum_breakpoints]
    if all([s >= 0.95 for s in optimum_scores]):
        print('All regions are high fidelity!')
    elif all([s >= 0.9 for s in optimum_scores]):
        print('Some regions are medium fidelity.')
    else:
        print('Some regions are low fidelity. Look closer')
        
    return optimum_breakpoints, optimum_overlaps, optimum_scores, optimum_lengths, oligo_array_indices

def generate_primer(DNA_seq,
                     Fwd=True,
                     extendtoCG=False,
                     smallest_primer_size=16,
                     largest_primer_size=30,
                     Tm=55):
    
    #Setup melting temperature arrays
    melt_temp_array = np.zeros(largest_primer_size-smallest_primer_size+1)
    
    if Fwd:
        DNA_seq_touse = DNA_seq
    else:
        DNA_seq_touse = DNA_seq.reverse_complement()
            
    #Make melting temperature arrays
    primer_length = 0
    for i in range(smallest_primer_size,largest_primer_size+1):
        melt_temp_array[i-smallest_primer_size] = mt.Tm_NN(DNA_seq_touse[0:i])
        
        #Pick F primer when Tm is first >F_Tm
        if (melt_temp_array[i-smallest_primer_size] >= Tm) & (primer_length==0):
            primer_length = i
    
    #If Tm isnt high enough after max bases, just set primer length to be max and hope it works
    if (primer_length == 0):
        primer_length = largest_primer_size
        
    if extendtoCG:
        while ((DNA_seq_touse[primer_length-1] == 'A') | (DNA_seq_touse[primer_length-1] == 'T')) & \
                    (primer_length < largest_primer_size):
            primer_length += 1
    
    return DNA_seq_touse[0:primer_length]

def make_all_mutations(region_name,
                       region,
                       region_flanks=[Seq(''),Seq('')],
                       nt_start=0, #zero-indexed!
                       wt_only=False,
                       synonymous=True,
                       stops='TAA',
                       all3ntdeletions=True,
                       codons_ranked_by_usage=codons_ranked_by_usage):
    
    oligo_array = {}
    #Check that region has size divisible by three
    if (len(region)/3 != len(region)//3) | (nt_start/3 != nt_start//3):
        print('Region is not translatable!')
        
    else:
        #add wt seq to oligo array
        oligo_name = region_name + '_WT'
        wt_seq = \
            region_flanks[0] + region + region_flanks[1]
        oligo_array[oligo_name] = wt_seq
        
        if not wt_only:
                    
            #loop over amino acids
            for j in range(0,len(region),3):

                #add all missense variants
                aa = region[j:(j+3)].translate()
                for aa_to in codons_ranked_by_usage.keys():
                    if aa_to != aa:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + str(aa_to)
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(codons_ranked_by_usage[aa_to][0]) + \
                            region[(j+3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                #add synonymous variant if True and if possible, 
                # using the most common codon that is NOT the codon in the gene
                if synonymous:
                    if len(codons_ranked_by_usage[aa]) > 1:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + str(aa)
                        possible_codons = codons_ranked_by_usage[aa].copy()
                        possible_codons.remove(region[j:(j+3)])
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(possible_codons[0]) + \
                            region[(j+3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                #add stops if true
                if stops:
                    oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + 'X'
                    seq_to_append = \
                        region_flanks[0] + \
                        region[0:j] + Seq(stops) + \
                        region[(j+3):] + \
                        region_flanks[1]
                    oligo_array[oligo_name] = seq_to_append

                #add all 3nt deletions if True
                if all3ntdeletions:
                    for k in range(0,3):
                        if j+k+3 <= len(region):
                            oligo_name = region_name + '_' + 'del' + str(nt_start+j+k+1)
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:(j+k)] + \
                                region[(j+k+3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append
        
    return oligo_array


def write_oligo_library(genes,
                        oligo_file='./oligo_test.csv',
                        primer_file='./primer_test.tsv',
                        gbl_file='./gbl_test.tsv',
                        primer_set_F=orthogonal_F,
                        primer_set_R=orthogonal_R,
                        codons_ranked_by_usage=codons_ranked_by_usage,
                        block_size_range=block_size_range, 
                        max_oligo_size=max_oligo_size,
                        slack=slack, 
                        empirical=sapI_overlap_empirical, 
                        codon_blacklist=codon_blacklist,
                        wt_only=False,
                        synonymous=True,
                        stops='TAA',
                        all3ntdeletions=True,
                        smallest_primer_size=16,
                        largest_primer_size=30,
                        Tm=55,
                        extendtoCG=True):
    
    #Split up primer set into F and R primers, cannot do more than 82 sublibraries
    oligo_primer_counter = 0
    oligo_array = {}
    amp_primers = {}
    gblocks = {}
    
    #Convert genes to Seq and genes to list
    gene_names = list(genes.keys())
    genes = [Seq(genes[gene_name]) for gene_name in gene_names]
    
    #SapI site sequence
    sapI_site = Seq('GCTCTTC')
    sapI_seqplusone = Seq('GCTCTTCC')
    pcr_capseq = Seq('GGCTAC') + sapI_seqplusone
    gbl_capseq_F = Seq('CCGCGTGATTACGAGTCG') + pcr_capseq
    gbl_capseq_R = Seq('GGGTTAGCAAGTGGCAGCCT') + pcr_capseq
    
    for r,gene in enumerate(genes):
        
        print('Processing gene ' + str(r+1))
        gene_name = gene_names[r]
        
        #exclude if gene size is not divisible by three
        if len(gene)/3 != len(gene)//3:
            print('Gene length is not divisible by 3!')
    
        #exclude if there is a SapI site in the gene
        elif any([True for kmer in build_kmers(gene, len(sapI_site)) if kmer==sapI_site]) | \
            any([True for kmer in build_kmers(gene.reverse_complement(), len(sapI_site)) if kmer==sapI_site]):
            print('Gene has SapI site!')
            
        else:
            
            print('Gene has no SapI site! Performing GoldenGate optimization...')
            
            #Optimize gene
            optimum_breakpoints, optimum_overlaps, optimum_scores, optimum_lengths, oligo_array_indices = \
            optimize_gene(gene, 
                      block_size_range=block_size_range, 
                      slack=slack, 
                      empirical=sapI_overlap_empirical, 
                      codon_blacklist=codon_blacklist)
            pprint.pprint({'Optimum Breakpoints': optimum_breakpoints, 
                   'Optimum Overlaps': optimum_overlaps, 
                   'Optimum Scores': optimum_scores})
            
            #add primers for gene_F and gene_R that are repeated constantly throughout the PCRs
            #note: should probably prevalidate these primers!
            F_primer = generate_primer(gene,
                                       Fwd=True,
                                       extendtoCG=extendtoCG,
                                       smallest_primer_size=smallest_primer_size,
                                       largest_primer_size=largest_primer_size,
                                       Tm=Tm)
            F_primer = pcr_capseq + F_primer
            amp_primers[gene_name+'_gene'+'_ampF'] = F_primer
            R_primer = generate_primer(gene,
                                       Fwd=False,
                                       extendtoCG=extendtoCG,
                                       smallest_primer_size=smallest_primer_size,
                                       largest_primer_size=largest_primer_size,
                                       Tm=Tm)
            R_primer = pcr_capseq + R_primer
            amp_primers[gene_name+'_gene'+'_ampR'] = R_primer
            
            #subset oligos and pcrs/gblocks
            for i,breakpoint in enumerate(optimum_breakpoints):
                
                #find indices of breakpoint that correspond to oligo vs need to be PCRed/gblock
                pcr_indices = [[j,j+1] for j in range(len(breakpoint)-1)]
                pcr_indices.remove(oligo_array_indices[i])
                
                #find mutagenic window of oligo
                oligo_breaks = [breakpoint[j] for j in oligo_array_indices[i]]
                oligo_mutagenic_window = [int(3*np.ceil((oligo_breaks[0]+3)/3)), int(3*np.floor(oligo_breaks[1]/3))]
                
                #add primers and gblocks
                for k,pcr_index in enumerate(pcr_indices):
                    piece_name = gene_name + '_block' + str(i+1) + '_s' + str(k+1)
                    pcr_breaks = [breakpoint[j] for j in pcr_index]
                    
                    if pcr_breaks[0] == breakpoint[0]: #Fragment beginning at gene start 
                        R_primer = generate_primer(gene[pcr_breaks[0]:(pcr_breaks[1]+3)],
                                                   Fwd=False,
                                                   extendtoCG=extendtoCG,
                                                   smallest_primer_size=smallest_primer_size,
                                                   largest_primer_size=largest_primer_size,
                                                   Tm=Tm)
                        R_primer = pcr_capseq + R_primer
                        amp_primers[piece_name+'_ampR'] = R_primer
                    elif pcr_breaks[1] == breakpoint[-1]: #Fragment ending at gene end
                        F_primer = generate_primer(gene[pcr_breaks[0]:(pcr_breaks[1]+3)],
                                                   Fwd=True,
                                                   extendtoCG=extendtoCG,
                                                   smallest_primer_size=smallest_primer_size,
                                                   largest_primer_size=largest_primer_size,
                                                   Tm=Tm)
                        F_primer = pcr_capseq + F_primer
                        amp_primers[piece_name+'_ampF'] = F_primer
                    else:
                        F_primer = generate_primer(gene[pcr_breaks[0]:(pcr_breaks[1]+3)],
                                                   Fwd=True,
                                                   extendtoCG=extendtoCG,
                                                   smallest_primer_size=smallest_primer_size,
                                                   largest_primer_size=largest_primer_size,
                                                   Tm=Tm)
                        F_primer = pcr_capseq + F_primer
                        R_primer = generate_primer(gene[pcr_breaks[0]:(pcr_breaks[1]+3)],
                                                   Fwd=False,
                                                   extendtoCG=extendtoCG,
                                                   smallest_primer_size=smallest_primer_size,
                                                   largest_primer_size=largest_primer_size,
                                                   Tm=Tm)
                        R_primer = pcr_capseq + R_primer
                        amp_primers[piece_name+'_ampF'] = F_primer
                        amp_primers[piece_name+'_ampR'] = R_primer
                    
                    gbl = gbl_capseq_F + gene[pcr_breaks[0]:(pcr_breaks[1]+3)] + gbl_capseq_R.reverse_complement()
                    gblocks[piece_name] = gbl
                    
                #add oligos to oligo array
                add_on_array = make_all_mutations(gene_name + '_block' + str(i+1),
                                   gene[oligo_mutagenic_window[0]:oligo_mutagenic_window[1]],
                                   region_flanks=[primer_set_F[oligo_primer_counter] + \
                                                  sapI_seqplusone + \
                                                  gene[oligo_breaks[0]:oligo_mutagenic_window[0]] ,
                                                  gene[oligo_mutagenic_window[1]:(oligo_breaks[1]+3)] + \
                                                  sapI_seqplusone.reverse_complement() + \
                                                  primer_set_R[oligo_primer_counter].reverse_complement()],
                                   nt_start=oligo_mutagenic_window[0],
                                   wt_only=wt_only,
                                   synonymous=synonymous,
                                   stops=stops,
                                   all3ntdeletions=all3ntdeletions,
                                   codons_ranked_by_usage=codons_ranked_by_usage)
                oligo_array.update(add_on_array)
                oligo_primer_counter += 1
                    
#                 #add WT
#                 oligo_name = gene_name + '_block' + str(i+1) + '_WT'
#                 wt_seq = \
#                     primer_set_F[oligo_primer_counter] + sapI_seqplusone + \
#                     gene[oligo_breaks[0]:(oligo_breaks[1]+3)] + \
#                     sapI_seqplusone.reverse_complement() + \
#                     primer_set_R[oligo_primer_counter].reverse_complement()
#                 oligo_array[oligo_name] = wt_seq
                
#                 if not wt_only:
                    
#                     #loop over amino acids
#                     for j in range(oligo_mutagenic_window[0],oligo_mutagenic_window[1],3):

#                         #add all missense variants
#                         aa = gene[j:(j+3)].translate()
#                         for aa_to in codons_ranked_by_usage.keys():
#                             if aa_to != aa:
#                                 oligo_name = gene_name + '_block' + str(i+1) + '_' + str(aa) + str(j//3+1) + str(aa_to)
#                                 seq_to_append = \
#                                     primer_set_F[oligo_primer_counter] + sapI_seqplusone + \
#                                     gene[oligo_breaks[0]:j] + Seq(codons_ranked_by_usage[aa_to][0]) + \
#                                     gene[(j+3):(oligo_breaks[1]+3)] + sapI_seqplusone.reverse_complement() + \
#                                     primer_set_R[oligo_primer_counter].reverse_complement()
#                                 oligo_array[oligo_name] = seq_to_append

#                         #add synonymous variant if True and if possible, 
#                         # using the most common codon that is NOT the codon in the gene
#                         if synonymous:
#                             if len(codons_ranked_by_usage[aa]) > 1:
#                                 oligo_name = gene_name + '_block' + str(i+1) + '_' + str(aa) + str(j//3+1) + str(aa)
#                                 possible_codons = codons_ranked_by_usage[aa].copy()
#                                 possible_codons.remove(gene[j:(j+3)])
#                                 seq_to_append = \
#                                     primer_set_F[oligo_primer_counter] + sapI_seqplusone + \
#                                     gene[oligo_breaks[0]:j] + Seq(possible_codons[0]) + \
#                                     gene[(j+3):(oligo_breaks[1]+3)] + sapI_seqplusone.reverse_complement() + \
#                                     primer_set_R[oligo_primer_counter].reverse_complement()
#                                 oligo_array[oligo_name] = seq_to_append

#                         #add stops if true
#                         if stops:
#                             oligo_name = gene_name + '_block' + str(i+1) + '_' + str(aa) + str(j//3+1) + 'X'
#                             seq_to_append = \
#                                 primer_set_F[oligo_primer_counter] + sapI_seqplusone + \
#                                 gene[oligo_breaks[0]:j] + Seq(stops) + \
#                                 gene[(j+3):(oligo_breaks[1]+3)] + sapI_seqplusone.reverse_complement() + \
#                                 primer_set_R[oligo_primer_counter].reverse_complement()
#                             oligo_array[oligo_name] = seq_to_append

#                         #add all 3nt deletions if True
#                         if all3ntdeletions:
#                             for k in range(0,3):
#                                 if j+k+3 <= oligo_breaks[1]:
#                                     oligo_name = gene_name + '_block' + str(i+1) + '_' + 'del' + str(j+k+1)
#                                     seq_to_append = \
#                                         primer_set_F[oligo_primer_counter] + sapI_seqplusone + \
#                                         gene[oligo_breaks[0]:j] + gene[j:j+k] + \
#                                         gene[(j+k+3):(oligo_breaks[1]+3)] + sapI_seqplusone.reverse_complement() + \
#                                         primer_set_R[oligo_primer_counter].reverse_complement()
#                                     oligo_array[oligo_name] = seq_to_append
                 
#                 oligo_primer_counter += 1
    
    #Check that max oligo is less than the max oligo length
    if sum([len(s)>max_oligo_size for s in oligo_array.values()]) == 0:
        print('All oligos are below the maximum 250bp!')
    else:
        print('Some oligos are TOO BIG!')
                
    #Remove any oligos with additional SapI sites
    bad_oligos = []
    for name,oligo in oligo_array.items():
        sapI_F = sum([True for kmer in build_kmers(oligo, len(sapI_site)) if kmer==sapI_site])
        sapI_R = sum([True for kmer in build_kmers(oligo.reverse_complement(), len(sapI_site)) if kmer==sapI_site])
        if (sapI_F != 1) | (sapI_R != 1):
            bad_oligos.append(name)
    for oligo_name in bad_oligos:
        del oligo_array[oligo_name]
    print(str(len(bad_oligos)) + ' oligos deleted due to errant SapI sites.')
    
    #Remove any duplicate oligos
    new_dict = {}
    seen_values = set()
    counter=0
    for key, value in oligo_array.items():
        if value not in seen_values:
            new_dict[key] = value
            seen_values.add(value)
        else:
            counter += 1
    print(str(counter) + ' oligos removed due to duplication.')
    oligo_array = new_dict
    del new_dict
    
    #write oligo array to file
    with open(oligo_file, 'w') as f:
        for key in oligo_array.keys():
            f.write("%s,%s\n"%(key,oligo_array[key]))
    f.close()
            
    #write primers to file
    primer_order_sheet = []
    for key in amp_primers.keys():
        primer_order_sheet.append(key + '\t' + \
                 str(amp_primers[key]) + \
                 '\t' + '25nm' + '\t' + 'STD\n')
    print(*primer_order_sheet)
    with open(primer_file, 'w') as f:
        for line in primer_order_sheet:
            f.write(line)
    f.close()
    
    #write gblocks to file
    gblock_order_sheet = []
    for key in gblocks.keys():
        gblock_order_sheet.append(key + '\t' + \
                 str(gblocks[key]) + '\n')
    print(*gblock_order_sheet)
    with open(gbl_file, 'w') as f:
        for line in gblock_order_sheet:
            f.write(line)
    f.close()
    
    return oligo_array,amp_primers,gblocks
                

In [10]:
optimize_gene(gene)


All regions are high fidelity!


([[0, 172, 477], [0, 159, 338, 477], [0, 313, 477]],
 [['ATG', 'GCT', 'ACG'], ['ATG', 'AGC', 'CTA', 'ACG'], ['ATG', 'GGG', 'ACG']],
 [0.9788060111199982, 0.973633231746607, 0.9783296401650228],
 [172, 182, 164],
 [[0, 1], [1, 2], [1, 2]])

In [90]:
oligo_array,amp_primers,gblocks = write_oligo_library({'SDHD':gene},
                                                      oligo_file='./sdhd_oligos.csv',
                                                      primer_file='./sdhd_primers.tsv',
                                                      gbl_file='./sdhd_gblocks.tsv')


Processing gene 1
Gene has no SapI site! Performing GoldenGate optimization...
All regions are high fidelity!
{'Optimum Breakpoints': [[0, 172, 477], [0, 159, 338, 477], [0, 313, 477]],
 'Optimum Overlaps': [['ATG', 'GCT', 'ACG'],
                      ['ATG', 'AGC', 'CTA', 'ACG'],
                      ['ATG', 'GGG', 'ACG']],
 'Optimum Scores': [0.9788060111199982, 0.973633231746607, 0.9783296401650228]}
All oligos are below the maximum 250bp!
4 oligos deleted due to errant SapI sites.
127 oligos removed due to duplication.
SDHD_block1_s1_ampF	GGCTACGCTCTTCCGCTCCAAGGCTGCATCTCTC	25nm	STD
 SDHD_block1_s1_ampR	GGCTACGCTCTTCCCGTGAGCTTCCACAGCATGG	25nm	STD
 SDHD_block2_s1_ampF	GGCTACGCTCTTCCATGGCGGTTCTCTGGAGGC	25nm	STD
 SDHD_block2_s1_ampR	GGCTACGCTCTTCCGCTCGGTGACAAGTGTATGTGC	25nm	STD
 SDHD_block2_s2_ampF	GGCTACGCTCTTCCCTATGTTCATGGGGATGCCTTGC	25nm	STD
 SDHD_block2_s2_ampR	GGCTACGCTCTTCCCGTGAGCTTCCACAGCATGG	25nm	STD
 SDHD_block3_s1_ampF	GGCTACGCTCTTCCATGGCGGTTCTCTGGAGGC	25nm	STD
 SDHD_block3

In [91]:
# nbn = \
# '''ATGTGGAAACTGCTGCCCGCCGCGGGCCCGGCAGGAGGAGAACCATACAGACTTTTGACTG
# GCGTTGAGTACGTTGTTGGAAGGAAAAACTGTGCCATTCTGATTGAAAATGATCAGTCGATCAG
# CCGAAATCATGCTGTGTTAACTGCTAACTTTTCTGTAACCAACCTGAGTCAAACAGATGAAATC
# CCTGTATTGACATTAAAAGATAATTCTAAGTATGGTACCTTTGTTAATGAGGAAAAAATGCAGA
# ATGGCTTTTCCCGAACTTTGAAGTCGGGGGATGGTATTACTTTTGGAGTGTTTGGAAGTAAATT
# CAGAATAGAGTATGAGCCTTTGGTTGCATGCTCCTCTTGTTTAGATGTCTCTGGGAAAACTGCT
# TTAAATCAAGCTATATTGCAACTTGGAGGATTTACTGTAAACAATTGGACAGAAGAATGCACTC
# ACCTTGTCATGGTATCAGTGAAAGTTACCATTAAAACAATATGTGCACTCATTTGTGGACGTCC
# AATTGTAAAGCCAGAATATTTTACTGAATTCCTGAAAGCAGTTGAGTCCAAGAAGCAGCCTCCA
# CAAATTGAAAGTTTTTACCCACCTCTTGATGAACCATCTATTGGAAGTAAAAATGTTGATCTGT
# CAGGACGGCAGGAAAGAAAACAAATCTTCAAAGGGAAAACATTTATATTTTTGAATGCCAAACA
# GCATAAGAAATTGAGTTCCGCAGTTGTCTTTGGAGGTGGGGAAGCTAGGTTGATAACAGAAGAG
# AATGAAGAAGAACATAATTTCTTTTTGGCTCCGGGAACGTGTGTTGTTGATACAGGAATAACAA
# ACTCACAGACCTTAATTCCTGACTGTCAGAAGAAATGGATTCAGTCAATAATGGATATGCTCCA
# AAGGCAAGGTCTTAGACCTATTCCTGAAGCAGAAATTGGATTGGCGGTGATTTTCATGACTACA
# AAGAATTACTGTGATCCTCAGGGCCATCCCAGTACAGGATTAAAGACAACAACTCCAGGACCAA
# GCCTTTCACAAGGCGTGTCAGTTGATGAAAAACTAATGCCAAGCGCCCCAGTGAACACTACAAC
# ATACGTAGCTGACACAGAATCAGAGCAAGCAGATACATGGGATTTGAGTGAAAGGCCAAAAGAA
# ATCAAAGTCTCCAAAATGGAACAAAAATTCAGAATGCTTTCACAAGATGCACCCACTGTAAAGG
# AGTCCTGCAAAACAAGCTCTAATAATAATAGTATGGTATCAAATACTTTGGCTAAGATGAGAAT
# CCCAAACTATCAGCTTTCACCAACTAAATTGCCAAGTATAAATAAAAGTAAAGATAGGGCTTCT
# CAGCAGCAGCAGACCAACTCCATCAGAAACTACTTTCAGCCGTCTACCAAAAAAAGGGAAAGGG
# ATGAAGAAAATCAAGAAATGTCTTCATGCAAATCAGCAAGAATAGAAACGTCTTGTTCTCTTTT
# AGAACAAACACAACCTGCTACACCCTCATTGTGGAAAAATAAGGAGCAGCATCTATCTGAGAAT
# GAGCCTGTGGACACAAACTCAGACAATAACTTATTTACAGATACAGATTTAAAATCTATTGTGA
# AAAATTCTGCCAGTAAATCTCATGCTGCAGAAAAGCTAAGATCAAATAAAAAAAGGGAAATGGA
# TGATGTGGCCATAGAAGATGAAGTATTGGAACAGTTATTCAAGGACACAAAACCAGAGTTAGAA
# ATTGATGTGAAAGTTCAAAAACAGGAGGAAGATGTCAATGTTAGAAAAAGGCCAAGGATGGATA
# TAGAAACAAATGACACTTTCAGTGATGAAGCAGTACCAGAAAGTAGCAAAATATCTCAAGAAAA
# TGAAATTGGGAAGAAACGTGAACTCAAGGAAGACTCACTATGGTCAGCTAAAGAAATATCTAAC
# AATGACAAACTTCAGGATGATAGTGAGATGCTTCCAAAAAAGCTGTTATTGACTGAATTTAGAT
# CACTGGTGATTAAAAACTCTACTTCCAGAAATCCATCTGGCATAAATGATGATTATGGTCAACT
# AAAAAATTTCAAGAAATTCAAAAAGGTCACATATCCTGGAGCAGGAAAACTTCCACACATCATT
# GGAGGATCAGATCTAATAGCTCATCATGCTCGAAAGAATACAGAACTAGAAGAGTGGCTAAGGC
# AGGAAATGGAGGTACAAAATCAACATGCAAAAGAAGAGTCTCTTGCTGATGATCTTTTTAGATA
# CAATCCTTATTTAAAAAGGAGAAGA'''
# nbn = nbn.replace('\n','').replace(' ','')
# nbn = nbn + 'ACG'
# optimize_gene(nbn)

In [ ]:
#SDHB with SapI site removal
sdhb=\
    '''ATGGCGGCGGTGGTCGCCCTCTCCTTGAGGCGCCGGTTGCCGGCCACAACCCTTGGC
    GGAGCCTGCCTGCAGGCCTCCCGAGGAGCCCAGACAGCTGCAGCCACAGCTCCCCGTATCAAGAAATTTG
    CCATCTATCGATGGGACCCAGACAAGGCTGGAGACAAACCTCATATGCAGACTTATGAAGTTGACCTTAA
    TAAATGTGGCCCCATGGTATTGGATGCTTTAATCAAGATTAAGAATGAAGTTGACTCTACTTTGACCTTC
    CGAAGATCATGCAGAGAAGGCATCTGTGGCTCTTGTGCAATGAACATCAATGGAGGCAACACTCTAGCTT
    GCACCCGAAGGATTGACACCAACCTCAATAAGGTCTCAAAAATCTACCCTCTTCCACACATGTATGTGAT
    AAAGGATCTTGTTCCCGATTTGAGCAACTTCTATGCACAGTACAAATCCATTGAGCCTTATTTGAAGAAG
    AAGGATGAATCTCAGGAAGGCAAGCAGCAGTATCTGCAGTCCATAGAGGAACGTGAGAAACTGGACGGGC
    TCTACGAGTGCATTCTCTGTGCCTGCTGTAGCACCAGCTGCCCCAGCTACTGGTGGAACGGAGACAAATA
    TCTGGGGCCTGCAGTTCTTATGCAGGCCTATCGCTGGATGATTGACTCCAGAGATGACTTCACAGAGGAG
    CGCCTGGCCAAGCTGCAGGACCCATTCTCTCTATACCGCTGCCACACCATCATGAACTGCACAAGGACCT
    GTCCTAAGGGTCTGAATCCAGGGAAAGCTATTGCAGAGATCAAGAAAATGATGGCAACCTATAAGGAGAA
    GAAAGCTTCAGTTTAACTGTTTCCATGCTAAACATGATTTATAACCAGCTCAGAGCTGAACATAATTTAT
    ATCTAATTTGAGTTCCTTTAAAGATCTTGGTTTTCCATGAATACAGCATGTATAATAAAAATTTTAAGAA
    ATAAATGTTATTCTACTTTATTAACAAAAAAAAAA'''
sdhb = sdhb.replace('\n','').replace(' ','')
sdhb = sdhb[0:280*3]+'ACG'
oligo_array,amp_primers,gblocks = write_oligo_library({'SDHB':sdhb},
                                                      oligo_file='./sdhb_oligos.csv',
                                                      primer_file='./sdhb_primers.tsv',
                                                      gbl_file='./sdhb_gblocks.tsv')


In [96]:
#add cyp2d6
sdhd=\
'ATGgcggttctctggaggctgagtgccgtttgcggtgccctaggaggccgagctctgttgcttcgaactccagtggtcagacctgctcatatctcagcatttcttcaggaccgacctatcccagaatggtgtggagtgcagcacatacacttgtcaccgagccaccattctggctccaaggctgcatctctccactggactagcgagagggttgtcagtgttttgctcctgggtctgcttccggctgcttatttgaatccttgctctgcgatggactattccctggctgcagccctcactcttcatggtcactggggccttggacaagttgttactgactatgttcatggggatgccttgcagaaagctgccaaggcagggcttttggcactttcagctttaacctttgctgggctttgctatttcaactatcacgatgtgggcatctgcaaagctgttgccatgctgtggaagctcacg'
sdhd=sdhd.upper()
cyp2d6=\
'ATGGGGCTGGAGGCTCTTGTCCCTCTGGCGGTCATCGTGGCTATCTTCCTCTTGCTTGTCGATCTGATGCACAGACGACAAAGGTGGGCAGCCCGGTATCCTCCTGGCCCTCTTCCCCTTCCCGGGCTGGGTAATCTGCTTCATGTGGACTTTCAGAATACGCCGTATTGTTTTGACCAGCTGAGAAGACGCTTTGGTGATGTTTTTAGCCTGCAATTGGCGTGGACACCCGTGGTCGTGCTGAATGGCCTGGCAGCGGTACGAGAAGCGTTGGTTACTCATGGGGAGGATACAGCCGACCGGCCCCCGGTACCCATTACTCAAATATTGGGGTTCGGACCCAGGTCCCAAGGTGTGTTTCTGGCAAGATATGGGCCTGCGTGGAGGGAACAACGCAGATTCTCTGTAAGCACCCTGAGGAACCTTGGTCTTGGAAAGAAGTCTCTCGAGCAGTGGGTCACCGAAGAAGCCGCGTGCTTGTGTGCAGCATTTGCGAATCACAGCGGTCGGCCTTTTCGCCCGAATGGCTTGCTGGATAAAGCTGTCTCTAATGTAATAGCTTCACTCACTTGTGGCCGAAGGTTTGAGTACGACGACCCTCGCTTCTTGCGACTGCTCGATTTGGCACAAGAGGGTCTCAAAGAGGAGAGCGGGTTCCTTCGCGAAGTTCTGAATGCTGTACCTGTGCTGTTGCATATCCCCGCGCTGGCAGGGAAAGTCCTTCGCTTCCAGAAGGCGTTTCTTACGCAACTCGATGAACTGCTTACGGAGCACAGGATGACATGGGACCCTGCTCAACCACCCCGAGACCTCACTGAAGCGTTTCTTGCTGAGATGGAAAAAGCCAAGGGAAATCCAGAAAGCTCTTTTAATGACGAGAACCTTCGGATTGTAGTGGCAGACCTCTTCAGTGCCGGTATGGTGACCACGTCTACCACTTTGGCCTGGGGCCTTCTCTTGATGATTTTGCATCCTGACGTCCAACGACGGGTTCAACAGGAGATTGACGATGTTATCGGCCAAGTTCGACGGCCTGAAATGGGGGATCAAGCCCACATGCCATATACTACGGCCGTCATCCACGAAGTCCAAAGGTTTGGCGACATCGTACCTTTGGGGGTCACGCACATGACCTCTCGCGACATAGAAGTGCAGGGCTTCAGGATTCCGAAGGGTACCACCTTGATAACCAACCTGTCTTCTGTCCTGAAGGACGAAGCAGTATGGGAGAAACCATTTCGCTTTCATCCGGAGCACTTCCTTGATGCGCAAGGGCATTTCGTCAAACCGGAGGCATTCCTTCCCTTCTCTGCAGGCCGCCGGGCCTGTCTTGGTGAACCTCTCGCACGAATGGAGCTTTTTCTTTTTTTCACTTCTTTGCTGCAGCACTTTTCCTTCTCTGTTCCAACTGGCCAGCCCCGCCCGTCTCACCACGGGGTATTCGCCTTTCTTGTAAGCCCGAGCCCCTACGAACTTTGCGCAGTCCCTAGAACG'


oligo_array,amp_primers,gblocks = write_oligo_library({'SDHD':sdhd, 'CYP2D6':cyp2d6},
                                                      oligo_file='./all_oligos.csv',
                                                      primer_file='./all_primers.tsv',
                                                      gbl_file='./all_gblocks.tsv')


Processing gene 1
Gene has no SapI site! Performing GoldenGate optimization...
All regions are high fidelity!
{'Optimum Breakpoints': [[0, 172, 477], [0, 159, 338, 477], [0, 313, 477]],
 'Optimum Overlaps': [['ATG', 'GCT', 'ACG'],
                      ['ATG', 'AGC', 'CTA', 'ACG'],
                      ['ATG', 'GGG', 'ACG']],
 'Optimum Scores': [0.9788060111199982, 0.973633231746607, 0.9783296401650228]}
Processing gene 2
Gene has no SapI site! Performing GoldenGate optimization...
All regions are high fidelity!
{'Optimum Breakpoints': [[0, 169, 1491],
                         [0, 164, 339, 1491],
                         [0, 329, 501, 1491],
                         [0, 486, 666, 1491],
                         [0, 653, 832, 1491],
                         [0, 821, 1000, 1491],
                         [0, 990, 1166, 1491],
                         [0, 1150, 1336, 1491],
                         [0, 1321, 1491]],
 'Optimum Overlaps': [['ATG', 'GTT', 'ACG'],
                      ['AT

In [66]:
#Order test primers

fh_sequence = 'AGAATATGATACCTTTGGTGAACTAAAGGTGCCAAATGATAAGTATTATGGCGCCCAGACCGTGAGATCTACGATGAACTTTAAGATTGGAGGTGTGACAGAACGCATGCCAACCCCAGTTATTAAAGCTTTTGGCATCTTGAAGCGAGCGGCCGCTGAAGTAAACCAGGATTATGGTCTTGATCCAAAGATTGCTAATGCAATAATGAA'
len(fh_sequence)

array = {}
for i in range(len(orthogonal_F[13:])):
    array['PrimerTest' + str(i+13)] = orthogonal_F[13+i] + fh_sequence + orthogonal_R[13+i].reverse_complement()

#write oligo array to file
    with open('./test_primers_oligos.csv', 'w') as f:
        for key in array.keys():
            f.write("%s,%s\n"%(key,array[key]))
            f.write("%s,%s\n"%(key,array[key]))
    f.close()

In [67]:
#Make raining's array
raining_array = make_all_mutations('TSC2_mini',
                       Seq('CACATCGCCACCCTGATGCCCACCAAGGACGTGGACAAGCACCGCTGCGACAAGAAGCGCCACCTGGGCAACGACTTTGTGTCCATTGTCTACAATGACTCCGGT'),
                       region_flanks=[Seq('TGCATAGTATCCCAACAGGGCAGTTCACCTACTGCTGGCACGATGACATCATGCAAGCCGTCTTC'),
                                      Seq('GTGGGGACTGTGCTTTGCAAGCCAGAGGGCTGGGGCTGGGTGATGACAGACTTGGGCTGGGCTAGGGGGG')],
                       nt_start=4857, #zero-indexed!
                       wt_only=False,
                       synonymous=True,
                       stops='TAA',
                       all3ntdeletions=True,
                       codons_ranked_by_usage=codons_ranked_by_usage)

#write raining's array to file
with open('./raining_array.csv', 'w') as f:
    for key in raining_array.keys():
        f.write("%s,%s\n"%(key,raining_array[key]))
f.close()

In [ ]:
# DO EVAN'S GENES

In [70]:
# HIST1H1E - ntag
optimize_gene(Seq('''ATGTCCGAGACTGCGCCTGCCGCGCCCGCTGCTCCGGCCCCTGCCGAGAAGACTCCCGTGA
AGAAGAAGGCCCGCAAGTCTGCAGGTGCGGCCAAGCGCAAAGCGTCTGGGCCCCCGGTGTCCGAGCTCATTACTAAAGCTGTT
GCCGCCTCCAAGGAGCGCAGCGGCGTATCTTTGGCCGCTCTCAAGAAAGCGCTGGCAGCCGCTGGCTATGACGTGGAGAAGAA
CAACAGCCGCATCAAGCTGGGTCTCAAGAGCCTGGTGAGCAAGGGCACCCTGGTGCAGACCAAGGGCACCGGCGCGTCGGGTT
CCTTCAAACTCAACAAGAAGGCGGCCTCTGGGGAAGCCAAGCCTAAGGCTAAAAAGGCAGGCGCGGCCAAGGCCAAGAAGCCAG
CAGGAGCGGCGAAGAAGCCCAAGAAGGCGACGGGGGCGGCCACCCCCAAGAAGTCCGCCAAGAAGACCCCAAAGAAGGCGAAGA
AGCCGGCTGCAGCTGCTGGAGCCAAAAAAGCGAAAAGCCCGAAAAAGGCGAAAGCAGCCAAGCCAAAAAAGGCGCCCAAGAGCCC
AGCGAAGGCCAAAGCAGTTAAACCCAAGGCGGCTAAACCAAAGACCGCCAAGCCCAAGGCAGCCAAGCCAAAGAAGGCGGCAGCCAAGAAAAAGTAG'''.replace('\n','')),
             block_size_range = [180,190])



All regions are high fidelity!


([[0, 191, 657], [0, 171, 366, 657], [0, 349, 552, 657], [0, 478, 657]],
 [['ATG', 'AGC', 'TAG'],
  ['ATG', 'TCT', 'GCA', 'TAG'],
  ['ATG', 'AGC', 'CCC', 'TAG'],
  ['ATG', 'AGC', 'TAG']],
 [0.987586931952799,
  0.9824329900400774,
  0.9823340840763735,
  0.987586931952799],
 [191, 198, 206, 179],
 [[0, 1], [1, 2], [1, 2], [1, 2]])

In [56]:
# RPS19 - ctag
optimize_gene(Seq('''ATGCCTGGAGTTACTGTAAAAGACGTGAACCAGCAGGAGTTCGTCAGAGCTCTGGCAGCCTTCCTCAAAAAGTC
CGGGAAGCTGAAAGTCCCCGAATGGGTGGATACCGTCAAGCTGGCCAAGCACAAAGAGCTTGCTCCCTACGATGAGAACTGGTTCTACACGCGAGC
TGCTTCCACAGCGCGGCACCTGTACCTCCGGGGTGGCGCTGGGGTTGGCTCCATGACCAAGATCTATGGGGGACGTCAGAGAAACGGCGTCATGCCC
AGCCACTTCAGCCGAGGCTCCAAGAGTGTGGCCCGCCGGGTCCTCCAAGCCCTGGAGGGGCTGAAAATGGTGGAAAAGGACCAAGATGGCGGCCGCA
AACTGACACCTCAGGGACAAAGAGATCTGGACAGAATCGCCGGACAGGTGGCAGCTGCCAACAAGAAGCATACG'''.replace('\n','')),
             block_size_range = [180,190])



All regions are high fidelity!


([[0, 191, 435], [0, 185, 384, 435], [0, 247, 435]],
 [['ATG', 'GTA', 'ACG'], ['ATG', 'GCA', 'AGA', 'ACG'], ['ATG', 'AGA', 'ACG']],
 [0.9787489256425429, 0.9737793681261416, 0.9787355027563043],
 [191, 202, 188],
 [[0, 1], [1, 2], [1, 2]])

In [57]:
# SPOP - ntag
optimize_gene(Seq('''ATGTCAAGGGTTCCAAGTCCTCCACCTCCGGCAGAAATGTCGAGTGGCCCCGTAGCTGAGAGTTGGTGCTACACACAGATCAAGGTAGTGA
AATTCTCCTACATGTGGACCATCAATAACTTTAGCTTTTGCCGGGAGGAAATGGGTGAAGTCATTAAAAGTTCTACATTTTCATCAGGAGCAAATGATAAACTGAAATGGTG
TTTGCGAGTAAACCCCAAAGGGTTAGATGAAGAAAGCAAAGATTACCTGTCACTTTACCTGTTACTGGTCAGCTGTCCAAAGAGTGAAGTTCGGGCAAAATTCAAATTCTCC
ATCCTGAATGCCAAGGGAGAAGAAACCAAAGCTATGGAGAGTCAACGGGCATATAGGTTTGTGCAAGGCAAAGACTGGGGATTCAAGAAATTCATCCGTAGAGATTTTCTTTT
GGATGAGGCCAACGGGCTTCTCCCTGATGACAAGCTTACCCTCTTCTGCGAGGTGAGTGTTGTGCAAGATTCTGTCAACATTTCTGGCCAGAATACCATGAACATGGTAAAGG
TTCCTGAGTGCCGGCTGGCAGATGAGTTAGGAGGACTGTGGGAGAATTCCCGGTTCACAGACTGCTGCTTGTGTGTTGCCGGCCAGGAATTCCAGGCTCACAAGGCTATCTTA
GCAGCTCGTTCTCCGGTTTTTAGTGCCATGTTTGAACATGAAATGGAGGAGAGCAAAAAGAATCGAGTTGAAATCAATGATGTGGAGCCTGAAGTTTTTAAGGAAATGATGTG
CTTCATTTACACGGGGAAGGCTCCAAACCTCGACAAAATGGCTGATGATTTGCTGGCAGCTGCTGACAAGTATGCCCTGGAGCGCTTAAAGGTCATGTGTGAGGATGCCCTCT
GCAGTAACCTGTCCGTGGAGAACGCTGCAGAAATTCTCATCCTGGCCGACCTCCACAGTGCAGATCAGTTGAAAACTCAGGCAGTGGATTTCATCAACTATCATGCTTCGGATG
TCTTGGAGACCTCTGGGTGGAAGTCAATGGTGGTGTCACATCCCCACTTGGTGGCTGAGGCATACCGCTCTCTGGCTTCAGCACAGTGCCCTTTTCTGGGACCCCCACGCAAA
CGCCTGAAGCAATCCTAG'''.replace('\n','')),
             block_size_range = [165,175])


All regions are high fidelity!


([[0, 170, 1122],
  [0, 160, 332, 1122],
  [0, 323, 504, 1122],
  [0, 486, 663, 1122],
  [0, 653, 836, 1122],
  [0, 813, 994, 1122],
  [0, 953, 1122]],
 [['ATG', 'TTC', 'TAG'],
  ['ATG', 'GTT', 'AGA', 'TAG'],
  ['ATG', 'TGC', 'AAC', 'TAG'],
  ['ATG', 'GTT', 'TCT', 'TAG'],
  ['ATG', 'AGC', 'GTA', 'TAG'],
  ['ATG', 'GAT', 'TCT', 'TAG'],
  ['ATG', 'AAC', 'TAG']],
 [0.9869819763078644,
  0.9829133479736809,
  0.9825411460676844,
  0.9829133479736809,
  0.982923062693338,
  0.9828616479886204,
  0.9875934989014593],
 [170, 175, 184, 180, 186, 184, 169],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [58]:
# AKT - ntag
optimize_gene(Seq('''ATGAGCGACGTGGCTATTGTGAAGGAGGGTTGGCTGCACAAACGAGGGGAGTACATCAAGACCTGGCGGCCACGCTACTTCCTCCTCA
AGAATGATGGCACCTTCATTGGCTACAAGGAGCGGCCGCAGGATGTGGACCAACGTGAGGCTCCCCTCAACAACTTCTCTGTGGCGCAGTGCCAGCTGATGAAGACGGA
GCGGCCCCGGCCCAACACCTTCATCATCCGCTGCCTGCAGTGGACCACTGTCATCGAACGCACCTTCCATGTGGAGACTCCTGAGGAGCGGGAGGAGTGGACAACCGCC
ATCCAGACTGTGGCTGACGGCCTCAAGAAGCAGGAGGAGGAGGAGATGGACTTCCGGTCGGGCTCACCCAGTGACAACTCAGGGGCTGAAGAGATGGAGGTGTCCCTGG
CCAAGCCCAAGCACCGCGTGACCATGAACGAGTTTGAGTACCTGAAGCTGCTGGGCAAGGGCACTTTCGGCAAGGTGATCCTGGTGAAGGAGAAGGCCACAGGCCGCTA
CTACGCCATGAAGATCCTCAAGAAGGAAGTCATCGTGGCCAAGGACGAGGTGGCCCACACACTCACCGAGAACCGCGTCCTGCAGAACTCCAGGCACCCCTTCCTCACA
GCCCTGAAGTACTCTTTCCAGACCCACGACCGCCTCTGCTTTGTCATGGAGTACGCCAACGGGGGCGAGCTGTTCTTCCACCTGTCCCGGGAGCGTGTGTTCTCCGAGG
ACCGGGCCCGCTTCTATGGCGCTGAGATTGTGTCAGCCCTGGACTACCTGCACTCGGAGAAGAACGTGGTGTACCGGGACCTCAAGCTGGAGAACCTCATGCTGGACAA
GGACGGGCACATTAAGATCACAGACTTCGGGCTGTGCAAGGAGGGGATCAAGGACGGTGCCACCATGAAGACCTTTTGCGGCACACCTGAGTACCTGGCCCCCGAGGTG
CTGGAGGACAATGACTACGGCCGTGCAGTGGACTGGTGGGGGCTGGGCGTGGTCATGTACGAGATGATGTGCGGTCGCCTGCCCTTCTACAACCAGGACCATGAGAAGC
TTTTTGAGCTCATCCTCATGGAGGAGATCCGCTTCCCGCGCACGCTTGGTCCCGAGGCCAAGTCCTTGCTTTCAGGGCTGCTCAAGAAGGACCCCAAGCAGAGGCTTGG
CGGGGGCTCCGAGGACGCCAAGGAGATCATGCAGCATCGCTTCTTTGCCGGTATCGTGTGGCAGCACGTGTACGAGAAGAAGCTCAGCCCACCCTTCAAGCCCCAGGTC
ACGTCGGAGACTGACACCAGGTATTTTGATGAGGAGTTCACGGCCCAGATGATCACCATCACACCACCTGACCAAGATGACAGCATGGAGTGTGTGGACAGCGAGCGCA
GGCCCCACTTCCCCCAGTTCTCCTACTCGGCCAGCGGCACGGCCTAG'''.replace('\n','')),
             block_size_range = [165,175])



All regions are high fidelity!


([[0, 167, 1440],
  [0, 156, 335, 1440],
  [0, 318, 500, 1440],
  [0, 491, 667, 1440],
  [0, 652, 834, 1440],
  [0, 817, 1001, 1440],
  [0, 983, 1160, 1440],
  [0, 1153, 1322, 1440],
  [0, 1276, 1440]],
 [['ATG', 'TGT', 'TAG'],
  ['ATG', 'AAC', 'GCA', 'TAG'],
  ['ATG', 'GCT', 'GAA', 'TAG'],
  ['ATG', 'GAT', 'TCT', 'TAG'],
  ['ATG', 'AGA', 'AAC', 'TAG'],
  ['ATG', 'GGG', 'GCT', 'TAG'],
  ['ATG', 'TGC', 'CCC', 'TAG'],
  ['ATG', 'AGA', 'GTT', 'TAG'],
  ['ATG', 'AGC', 'TAG']],
 [0.9874119857668283,
  0.9825411460676844,
  0.9818195501853842,
  0.9828616479886204,
  0.9829133479736809,
  0.9823340840763735,
  0.981863143258296,
  0.9829133479736809,
  0.987586931952799],
 [167, 182, 185, 179, 185, 187, 180, 172, 164],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [68]:
# CFTR portion - c tag
optimize_gene(Seq('''ATGGAGAATGTAACAGCCTTCTGGGAGGAGGGATTTGGGGAATTATTTGAGAAAGCAAAACAAAACAATAACAATAG
AAAAACTTCTAATGGTGATGACAGCCTCTTCTTCAGTAATTTCTCACTTCTTGGTACTCCTGTCCTGAAAGATATTAATTTCAAGATAGAAAGAGGAC
AGTTGTTGGCGGTTGCTGGATCCACTGGAGCAGGCAAGACTTCACTTCTAATGGTGATTATGGGAGAACTGGAGCCTTCAGAGGGTAAAATTAAGCAC
AGTGGAAGAATTTCATTCTGTTCTCAGTTTTCCTGGATTATGCCTGGCACCATTAAAGAAAATATCATCTTTGGTGTTTCCTATGATGAATATAGATA
CAGAAGCGTCATCAAAGCATGCCAACTAGAAGAGGACATCTCCAAGTTTGCAGAGAAAGACAATATAGTTCTTGGAGAAGGTGGAATCACACTGAGTG
GAGGTCAACGAGCAAGAATTAGC'''.replace('\n','')),
             block_size_range = [175,185])



All regions are high fidelity!


([[0, 178, 489], [0, 165, 353, 489], [0, 307, 489]],
 [['ATG', 'TGT', 'AGC'], ['ATG', 'GAA', 'CTA', 'AGC'], ['ATG', 'GGA', 'AGC']],
 [0.9938372832702521, 0.9883744528802465, 0.9931496053266099],
 [178, 191, 182],
 [[0, 1], [1, 2], [1, 2]])